# Assignment 3


This notebook will demonstrate using python libraries for interactive dashboards using Panel + Plotly Express. I choose those librabries becuase 
they interface similar to matplotlib and Seaborn. 

Becauase i have a PHEV and I'm always looking for EV charging stations. I have chosen the U.S. Alternative Fueling Stations dataset https://catalog.data.gov/dataset/alternative-fueling-stations 
and I will narrow it to just Michigan stations. it was challenging becuase while I was building and testing I ran out of my API call allotment so I had to download the file


In [ ]:
# One-time Vocareum setup only.

# %pip install --target /voc/work/work/panel_packages --no-deps panel param pyviz-comms nh3

# %pip install --target /voc/work/work/panel_packages --no-deps \
#     markdown markdown-it-py mdit-py-plugins linkify-it-py \
#     mdurl uc-micro-py 
#%pip install \
#    --target /voc/work/work/panel_packages \
#    --no-deps \
#    jupyter_bokeh

was not not rendering properly, had to check Panel/Bokeh version mismatch possily created by installing Panel separately.

In [ ]:
import sys
from pathlib import Path
from urllib.parse import urlencode
from getpass import getpass

import numpy as np
import pandas as pd
import plotly
import plotly.express as px

# Panel was installed locally because Vocareum's
# system Python directory is read-only.
panel_directory = Path(
    "/voc/work/work/panel_packages"
)

if str(panel_directory) not in sys.path:
    sys.path.append(str(panel_directory))

import bokeh
import panel as pn
import param
import pyviz_comms

pn.extension(
    "plotly",
    "tabulator",
    sizing_mode="stretch_width"
)





In [ ]:
versions = {
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "Plotly": plotly.__version__,
    "Panel": pn.__version__,
    "Bokeh": bokeh.__version__,
    "Param": param.__version__,
    "PyViz Comms": pyviz_comms.__version__
}

print("Jupyter Bokeh:", jupyter_bokeh.__version__)
print("Panel:", pn.__version__)
print("Communication mode:", pn.config.comms)

pd.Series(
    versions,
    name="Version"
).to_frame()

In [ ]:
data_file = Path("michigan_ev_charging_raw.csv")

if data_file.exists():
    print("Loading saved Michigan dataset")

    ev_mi = pd.read_csv(
        data_file,
        low_memory=False
    )

else:
    print("Local dataset not found; downloading it once")

    api_key = getpass("Enter your NLR API key: ")

    parameters = {
        "api_key": api_key,
        "fuel_type": "ELEC",
        "state": "MI",
        "status": "E",
        "limit": "all"
    }

    base_url = (
        "https://"
        + "developer.nlr.gov/api/alt-fuel-stations/v1.csv"
    )

    url = base_url + "?" + urlencode(parameters)

    ev_mi = pd.read_csv(
        url,
        low_memory=False
    )

    ev_mi.to_csv(
        data_file,
        index=False
    )

    print("Dataset downloaded and saved")

print(f"Rows: {len(ev_mi):,}")
print(f"Columns: {len(ev_mi.columns):,}")


In [ ]:
""" Had to delete this code to get it from the URL because I ran into API limit
#TO DO Once it is working and tested, go back to loading from URL not local download


# load the USAFS dataset and save locally one time

parameters = {
    "api_key": "DEMO_KEY",
    "fuel_type": "ELEC",
    "state": "MI",
#    "city": "Detroit",
    "status": "E",
    "limit": "all"
}

# Split to prevent the notebook interface from turning it into Markdown
base_url = (
    "https://"
    + "developer.nlr.gov/api/alt-fuel-stations/v1.csv"
)

url = base_url + "?" + urlencode(parameters)

print(url)

ev_mi = pd.read_csv(url, low_memory=False)

print(f"Rows: {len(ev_mi):,}")
print(f"Columns: {len(ev_mi.columns):,}")

ev_mi.head()
"""

now test that it was loaded  for just Michigan

In [ ]:
print(f"Rows: {ev_mi.shape[0]:,}")
print(f"Columns: {ev_mi.shape[1]:,}")

print(ev_mi["State"].value_counts(dropna=False))

Get the columns and shrink the columns to the ones we want

In [ ]:

columns_to_keep = [
    "Fuel Type Code",
    "Station Name",
    "Street Address",
    "City",
    "State",
    "ZIP",
    "Status Code",
    "Access Code",
    "Access Days Time",
    "EV Pricing",
    "EV Level1 EVSE Num",
    "EV Level2 EVSE Num",
    "EV DC Fast Count",
    "EV Network",
    "EV Network Web",
    "EV Connector Types",
    "Latitude",
    "Longitude",
    "Open Date",
    "ID"
]

ev_mi = ev_mi[columns_to_keep].copy()

ev_mi.head()


Prepare dates and charging-port measures”—the API already selects available stations.

In [ ]:
ev_mi["Open Date"] = pd.to_datetime(
    ev_mi["Open Date"],
    errors="coerce"
)

port_columns = [
    "EV Level1 EVSE Num",
    "EV Level2 EVSE Num",
    "EV DC Fast Count"
]

ev_mi[port_columns] = (
    ev_mi[port_columns]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

ev_mi["Total Ports"] = ev_mi[port_columns].sum(axis=1)

ev_mi["Open Year"] = ev_mi["Open Date"].dt.year.astype("Int64")

In [ ]:
#convert zip code to strings
ev_mi["ZIP"] = (
    ev_mi["ZIP"]
    .astype("string")
    .str.zfill(5)
)
#see a smaple of what we have
ev_mi[
    [
        "Station Name",
        "City",
        "EV Network",
        "EV Level2 EVSE Num",
        "EV DC Fast Count",
        "Total Ports"
    ]
].head(10)

In [ ]:
ev_mi.loc[
    ev_mi["Total Ports"] == 0,
    [
        "Station Name",
        "City",
        "EV Level1 EVSE Num",
        "EV Level2 EVSE Num",
        "EV DC Fast Count"
    ]
]

In [ ]:
#make sure all the years are valis
ev_mi.loc[
    ev_mi["Open Year"].notna()
    & ~ev_mi["Open Year"].between(1990, 2026),
    ["Station Name", "City", "Open Date", "Open Year"]
]

In [ ]:
print("Rows:", f"{len(ev_mi):,}")
print("Columns:", len(ev_mi.columns))
print("Duplicate columns:", ev_mi.columns.duplicated().sum())
print("Duplicate station IDs:", ev_mi["ID"].duplicated().sum())
print("Missing coordinates:", ev_mi[["Latitude", "Longitude"]].isna().any(axis=1).sum())
print("Stations with zero ports:", (ev_mi["Total Ports"] == 0).sum())
print("Invalid opening years:", (~ev_mi["Open Year"].between(1990, 2026)).sum())


In [ ]:
ev_mi["EV Connector Types"].value_counts().head(20)

In [ ]:
ev_connectors = (
    ev_mi[
        [
            "ID",
            "Station Name",
            "City",
            "EV Network",
            "EV Connector Types"
        ]
    ]
    .assign(
        Connector=lambda df: (
            df["EV Connector Types"]
            .fillna("")
            .str.split()
        )
    )
    .explode("Connector")
    .query("Connector != ''")
    .reset_index(drop=True)
)

In [ ]:

connector_counts = (
    ev_connectors
    .groupby("Connector", as_index=False)
    .agg(Stations=("ID", "nunique"))
    .sort_values("Stations")
)

#build the chart to show number of connectrs by type
fig = px.bar(
    connector_counts,
    x="Stations",
    y="Connector",
    orientation="h",
    title="Michigan EV Stations by Connector Type",
    text="Stations"
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside"
)

fig.show()

Now that we have the data cleaned and the df organized let's build some widgets and then panels

In [ ]:
#create the widgets to use
city_filter = pn.widgets.Select(
    name="City",
    options=["All"] + sorted(
        ev_mi["City"].dropna().unique().tolist()
    ),
    value="All"
)

network_filter = pn.widgets.Select(
    name="Charging Network",
    options=["All"] + sorted(
        ev_mi["EV Network"].dropna().unique().tolist()
    ),
    value="All"
)

access_filter = pn.widgets.Select(
    name="Access Type",
    options=["All"] + sorted(
        ev_mi["Access Code"].dropna().unique().tolist()
    ),
    value="All"
)

connector_options = sorted({
    connector
    for value in ev_mi["EV Connector Types"].dropna()
    for connector in value.split()
})

connector_filter = pn.widgets.Select(
    name="Connector Type",
    options=["All"] + connector_options,
    value="All"
)

minimum_year = int(
    ev_mi["Open Year"].dropna().min()
)

maximum_year = int(
    ev_mi["Open Year"].dropna().max()
)

year_filter = pn.widgets.IntRangeSlider(
    name="Opening Year",
    start=minimum_year,
    end=maximum_year,
    value=(minimum_year, maximum_year),
    step=1
)

minimum_ports_filter = pn.widgets.IntSlider(
    name="Minimum Total Ports",
    start=0,
    end=int(ev_mi["Total Ports"].max()),
    value=0,
    step=1
)

dc_fast_filter = pn.widgets.Checkbox(
    name="DC Fast Charging Only",
    value=False
)


In [ ]:
#test the widget controls
"""
pn.Column(
    city_filter,
    network_filter,
    access_filter,
    connector_filter,
    year_filter,
    minimum_ports_filter,
    dc_fast_filter
)
"""

In [ ]:
#reusable filter function

def filter_stations(
    city,
    network,
    access_type,
    connector,
    years,
    minimum_ports,
    dc_fast_only
):
    data = ev_mi.copy()

    if city != "All":
        data = data[
            data["City"].eq(city)
        ]

    if network != "All":
        data = data[
            data["EV Network"].eq(network)
        ]

    if access_type != "All":
        data = data[
            data["Access Code"].eq(access_type)
        ]

    if connector != "All":
        connector_lists = (
            data["EV Connector Types"]
            .fillna("")
            .str.split()
        )

        data = data[
            connector_lists.apply(
                lambda available: connector in available
            )
        ]

    if years != (minimum_year, maximum_year):
        year_mask = (
            data["Open Year"]
            .between(years[0], years[1])
            .fillna(False)
        )

        data = data[year_mask]

    data = data[
        data["Total Ports"] >= minimum_ports
    ]

    if dc_fast_only:
        data = data[
            data["EV DC Fast Count"] > 0
        ]

    return data
    

In [ ]:
#Create KPIS
def make_kpis(data):
    station_count = len(data)
    total_ports = int(data["Total Ports"].sum())
    dc_fast_ports = int(data["EV DC Fast Count"].sum())
    city_count = data["City"].nunique()

    public_percentage = (
        data["Access Code"].eq("public").mean() * 100
        if station_count > 0
        else 0
    )

    def make_card(label, value):
        return pn.pane.HTML(
            f"""
            <div style="
                background:white;
                border:1px solid #d0d7de;
                border-radius:8px;
                padding:14px;
                min-width:145px;
                text-align:center;
            ">
                <div style="
                    color:#57606a;
                    font-size:13px;
                ">
                    {label}
                </div>

                <div style="
                    color:#006b5e;
                    font-size:26px;
                    font-weight:600;
                ">
                    {value}
                </div>
            </div>
            """
        )

    return pn.FlexBox(
        make_card("Stations", f"{station_count:,}"),
        make_card("Total Ports", f"{total_ports:,}"),
        make_card("DC Fast Ports", f"{dc_fast_ports:,}"),
        make_card("Cities", f"{city_count:,}"),
        make_card("Public Access", f"{public_percentage:.1f}%"),
        justify_content="space-between"
    )

In [ ]:
#function to make the maps
def make_map(data):
    if data.empty:
        figure = px.scatter(
            title="No stations match the selected filters"
        )

        figure.add_annotation(
            text="Change or reset one or more filters.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False
        )

        return figure

    map_data = data.copy()

    map_data["Marker Size"] = (
        map_data["Total Ports"].clip(lower=1)
    )

    figure = px.scatter_map(
        map_data,
        lat="Latitude",
        lon="Longitude",
        color="EV Network",
        size="Marker Size",
        size_max=20,
        hover_name="Station Name",
        hover_data={
            "City": True,
            "Access Code": True,
            "EV Connector Types": True,
            "EV Level2 EVSE Num": True,
            "EV DC Fast Count": True,
            "Total Ports": True,
            "Latitude": False,
            "Longitude": False,
            "Marker Size": False
        },
        center={
            "lat": 44.3,
            "lon": -85.6
        },
        zoom=5.2,
        map_style="open-street-map",
        height=600,
        title="Michigan EV Charging Stations"
    )

    figure.update_layout(
        margin={
            "l": 10,
            "r": 10,
            "t": 50,
            "b": 10
        },
        legend_title="Network"
    )

    return figure


In [ ]:
#network comparison
def make_network_chart(data):
    if data.empty:
        return px.bar(
            title="No network data matches the filters"
        )

    network_data = (
        data
        .groupby("EV Network", as_index=False)
        .agg(
            Stations=("ID", "nunique"),
            Total_Ports=("Total Ports", "sum"),
            DC_Fast_Ports=("EV DC Fast Count", "sum")
        )
        .nlargest(15, "Stations")
        .sort_values("Stations")
    )

    figure = px.bar(
        network_data,
        x="Stations",
        y="EV Network",
        orientation="h",
        text="Stations",
        hover_data={
            "Total_Ports": ":,",
            "DC_Fast_Ports": ":,"
        },
        title="Largest Charging Networks"
    )

    figure.update_traces(
        texttemplate="%{text:,}",
        textposition="outside"
    )

    figure.update_layout(
        xaxis_title="Number of Stations",
        yaxis_title="Charging Network",
        showlegend=False,
        height=500,
        margin={
            "l": 10,
            "r": 60,
            "t": 50,
            "b": 40
        }
    )

    return figure



In [ ]:
#growth chart
def make_growth_chart(data):
    dated_data = data.dropna(
        subset=["Open Year"]
    ).copy()

    if dated_data.empty:
        return px.line(
            title="No opening-date data matches the filters"
        )

    annual_data = (
        dated_data
        .groupby("Open Year", as_index=False)
        .agg(
            Stations_Opened=("ID", "nunique"),
            Ports_Added=("Total Ports", "sum")
        )
        .sort_values("Open Year")
    )

    annual_data["Open Year"] = (
        annual_data["Open Year"].astype(int)
    )

    figure = px.line(
        annual_data,
        x="Open Year",
        y="Stations_Opened",
        markers=True,
        hover_data={
            "Ports_Added": ":,"
        },
        title="Charging Stations Opened by Year"
    )

    figure.update_layout(
        xaxis_title="Opening Year",
        yaxis_title="Stations Opened",
        height=500,
        margin={
            "l": 10,
            "r": 20,
            "t": 50,
            "b": 40
        }
    )

    return figure



In [ ]:
#interactive table
table_columns = [
    "Station Name",
    "City",
    "Street Address",
    "EV Network",
    "Access Code",
    "EV Connector Types",
    "EV Level2 EVSE Num",
    "EV DC Fast Count",
    "Total Ports",
    "Open Date"
]

def make_table(data):
    table_data = (
        data[table_columns]
        .sort_values(
            ["City", "Station Name"]
        )
        .reset_index(drop=True)
    )

    return pn.widgets.Tabulator(
        table_data,
        pagination="local",
        page_size=15,
        header_filters=True,
        disabled=True,
        show_index=False,
        height=450,
        sizing_mode="stretch_width"
    )



In [ ]:
# Reset all dashboard filters to their default values
reset_button = pn.widgets.Button(
    name="Reset All Filters",
    button_type="primary",
    width_policy="max"
)

def reset_filters(event):
    city_filter.value = "All"
    network_filter.value = "All"
    access_filter.value = "All"
    connector_filter.value = "All"

    year_filter.value = (
        minimum_year,
        maximum_year
    )

    minimum_ports_filter.value = 0
    dc_fast_filter.value = False

reset_button.on_click(reset_filters)


In [ ]:

@pn.depends(
    city=city_filter.param.value,
    network=network_filter.param.value,
    access_type=access_filter.param.value,
    connector=connector_filter.param.value,
    years=year_filter.param.value,
    minimum_ports=minimum_ports_filter.param.value,
    dc_fast_only=dc_fast_filter.param.value
)
def reactive_content(
    city,
    network,
    access_type,
    connector,
    years,
    minimum_ports,
    dc_fast_only
):
    data = filter_stations(
        city=city,
        network=network,
        access_type=access_type,
        connector=connector,
        years=years,
        minimum_ports=minimum_ports,
        dc_fast_only=dc_fast_only
    )

    return pn.Column(
        make_kpis(data),

        pn.Card(
            make_map(data),
            title="Station Map",
            collapsed=False
        ),

        pn.Row(
            pn.Card(
                make_network_chart(data),
                title="Network Comparison"
            ),
            pn.Card(
                make_growth_chart(data),
                title="Growth Over Time"
            )
        ),

        pn.Card(
            make_table(data),
            title=f"Charging-Station Details — {len(data):,} stations"
        ),

        sizing_mode="stretch_width"
    )

In [ ]:
#put it together into a dashboard

filter_sidebar = pn.Card(
    pn.pane.Markdown(
        """
        Select one or more filters. The KPIs, charts and
        table update automatically.
        """
    ),
    city_filter,
    network_filter,
    access_filter,
    connector_filter,
    year_filter,
    minimum_ports_filter,
    dc_fast_filter,
    reset_button,
    title="Filters",
    width=320,
    sizing_mode="fixed",
    collapsed=False
)

dashboard = pn.Column(
    pn.pane.Markdown(
        """
        # Michigan EV Charging Infrastructure

        Explore currently available Michigan charging stations
        by location, network, access, connector type, opening
        year and charging capability.
        """
    ),

    pn.Row(
        filter_sidebar,
        reactive_content,
        sizing_mode="stretch_width"
    ),

    sizing_mode="stretch_width"
)

dashboard